# KnoArbor Ingest Flow Walkthrough

这个 notebook 用一个固定中文案例完整跑一遍 KnoArbor ingest，并逐步展示每个模块的输入输出。

目标：

- 看清 `Connector -> SourceDocument -> Checkpoint -> Redaction -> Segmentation -> Semantic Agents -> Write -> Scoped Lint -> Report` 的链路。
- 查看 5 个语义 contract 的真实输入输出：`source_normalize`、`wiki_atom_extract`、`wiki_page_plan`、`wiki_draft_compile`、`ingest_draft_review`。
- 使用临时 vault，不触碰你的真实知识库。

运行方式：按顺序执行全部 cell。需要 `.env` 或环境变量里有 `DEEPSEEK_API_KEY`。

## 0. 流程地图

```text
Markdown file
  -> Connector Discovery
  -> RawSource
  -> SourceDocument
  -> Checkpoint Decision
  -> Privacy Redaction
  -> Source Segmentation
  -> source_normalize Agent
  -> SourceDigest Builder
  -> wiki_atom_extract Agent
  -> Wiki Context Retrieval
  -> wiki_page_plan Agent
  -> IngestCompileContext
  -> wiki_draft_compile Agent
  -> ingest_draft_review Agent
  -> Quality Gate
  -> Wiki Write
  -> Knowledge Atom Index
  -> Scoped Lint
  -> Checkpoint Commit
  -> Report / Ledger / Token Ledger
```

In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from datetime import datetime
from pathlib import Path
from typing import Any

import yaml
from IPython.display import JSON, Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').exists():
    raise RuntimeError('请从 KnoArbor 项目根目录启动 Jupyter。当前目录不是项目根目录。')

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from knoarbor.core.config import load_config, load_env_file
from knoarbor.pipelines.ingest import IngestPipeline
from knoarbor.pipelines.source import SourcePipeline
from knoarbor.semantic.factory import build_semantic_runner
from knoarbor.semantic.ingest_workflow import IngestSemanticWorkflow
from knoarbor.semantic.runner import SemanticRunner
from knoarbor.storage.wiki_init import init_wiki_vault
from knoarbor.storage.wiki_paths import content_root

load_env_file(ROOT / '.env')
if not os.environ.get('DEEPSEEK_API_KEY'):
    raise RuntimeError('未发现 DEEPSEEK_API_KEY。请先在 .env 或环境变量中配置。')

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = ROOT / 'tmp' / f'notebook-ingest-flow-{RUN_ID}'
INPUT_DIR = RUN_DIR / 'input'
VAULT_DIR = RUN_DIR / 'vault'
ARTIFACTS_DIR = RUN_DIR / 'artifacts'
for directory in (INPUT_DIR, VAULT_DIR, ARTIFACTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

display(Markdown(f'**Run directory**: `{RUN_DIR}`'))

## 1. 固定案例：个人 AI Wiki 的维护原则

这个案例覆盖 KnoArbor 自身最核心的知识对象：Raw、Source、Wiki page、Graph、自动维护、RAG 对比。它不长，适合观察每个模块如何拆解和编译。

In [ ]:
SAMPLE_TEXT = '''# 个人 AI Wiki 的维护原则

个人 AI Wiki 不是简单保存聊天记录，而是把分散的对话、笔记和文档编译成可追溯、可维护的知识网络。Raw source 应保持原始性，用于追溯和审计；Wiki page 则是经过整理后的长期知识对象。

维护原则包括：第一，来源和生成页面要分离，避免把临时材料误认为稳定知识；第二，页面应围绕稳定知识对象组织，而不是围绕一次对话或一个文件机械生成；第三，页面之间需要通过关系、引用和反向链接形成网络；第四，维护流程要尽量自动化，但对高风险修改保留审计能力。

和传统 RAG 相比，AI Wiki 更强调沉淀后的页面质量，而不是一次性召回大量 chunk。RAG 适合快速问答和临时检索，AI Wiki 更适合长期积累、复用、审计和持续维护。
'''

SAMPLE_PATH = INPUT_DIR / 'personal_ai_wiki_maintenance.md'
SAMPLE_PATH.write_text(SAMPLE_TEXT, encoding='utf-8')
display(Markdown('### 样本原文'))
display(Markdown(SAMPLE_TEXT))
display(Markdown(f'样本文件：`{SAMPLE_PATH}`'))

## 2. 临时配置

这里只启用 `markdown` connector，关闭其他来源。Vault 写入 `tmp/`，不会修改真实知识库。

In [ ]:
CONFIG_PATH = RUN_DIR / 'config.yaml'
config_data = {
    'project': {'name': 'Notebook Ingest Flow', 'host_project_root': str(ROOT)},
    'config_version': 1,
    'vaults': {'default': 'notebook', 'profiles': {'notebook': {'name': 'Notebook Ingest Flow', 'path': str(VAULT_DIR)}}},
    'vault': {'path': str(VAULT_DIR)},
    'server': {'host': '127.0.0.1', 'port': 8000},
    'models': {
        'default_provider': 'deepseek',
        'default_max_tokens': 30000,
        'request_timeout_seconds': 600,
        'retry': {'enabled': True, 'max_attempts': 2, 'backoff_seconds': 2, 'retry_on_invalid_output': True},
        'providers': {
            'deepseek': {
                'base_url': 'https://api.deepseek.com',
                'api_key_env': 'DEEPSEEK_API_KEY',
                'model': 'deepseek-v4-flash',
                'json_mode': True,
                'verify_tls': True,
                'tls_ca_file': None,
            }
        },
    },
    'document_processing': {'mineru': {'enabled': False}},
    'connectors': {
        'markdown': {
            'enabled': True,
            'settings': {
                'roots': [str(SAMPLE_PATH)],
                'recursive': False,
                'raw_output_dir': str(VAULT_DIR / 'raw' / 'notes'),
                'preserve_relative_paths': True,
            },
        },
        'codex': {'enabled': False, 'settings': {}},
        'hermes': {'enabled': False, 'settings': {}},
        'openclaw': {'enabled': False, 'settings': {}},
        'claude_code': {'enabled': False, 'settings': {}},
    },
    'ingest': {
        'candidate_limit': 8,
        'materialized_page_limit': 8,
        'max_chars_per_materialized_page': 6000,
        'auto_scoped_lint': True,
        'auto_apply_safe_lint_fixes': True,
        'segmentation': {
            'enabled': True,
            'max_chars_per_segment': 18000,
            'soft_chars_per_segment': 12000,
            'overlap_chars': 1200,
            'max_segments_per_source': 20,
            'min_segment_chars': 0,
        },
        'recovery': {'enabled': True, 'execution_ledger_path': 'maintenance/ingest_execution_ledger.jsonl'},
        'concurrency': {'max_concurrent_sources': 1},
    },
    'lint': {'default_scope': 'latest-ingest', 'scoped_include_related': True, 'include_neighbors': True, 'include_global_checks': True},
    'privacy': {
        'redaction_enabled': True,
        'redact_emails': True,
        'redact_phone_numbers': True,
        'redact_api_keys': True,
        'redact_private_keys': True,
        'redact_platform_ids': True,
        'redact_local_paths': True,
        'redact_source_paths_in_pages': True,
        'redact_private_ips': False,
        'custom_terms': [],
    },
}
CONFIG_PATH.write_text(yaml.safe_dump(config_data, allow_unicode=True, sort_keys=False), encoding='utf-8')
init_wiki_vault(VAULT_DIR, force=False)
config = load_config(CONFIG_PATH)

display(Markdown(f'配置文件：`{CONFIG_PATH}`'))
display(Markdown(f'临时 vault：`{VAULT_DIR}`'))
display(JSON(config.model_dump(mode='json'), expanded=False))

## 3. Connector / RawSource / SourceDocument

这一步是确定性逻辑，不调用模型：发现文件、读取文件、转成统一 `SourceDocument`。

In [ ]:
source_pipeline = SourcePipeline()
source_result = source_pipeline.run('markdown', config.connectors['markdown'])
(ARTIFACTS_DIR / 'source_pipeline_output.json').write_text(
    json.dumps(source_result.model_dump(mode='json'), ensure_ascii=False, indent=2),
    encoding='utf-8',
)

source_item = source_result.items[0]
display(Markdown('### SourceRef'))
display(JSON(source_item.ref.model_dump(mode='json'), expanded=True))
display(Markdown('### RawSource'))
display(JSON(source_item.raw.model_dump(mode='json'), expanded=True))
display(Markdown('### SourceDocument'))
display(JSON(source_item.document.model_dump(mode='json'), expanded=False))

## 4. 记录语义 Agent 输入输出

下面的 `RecordingRunner` 包住真实 `SemanticRunner`。每个 contract 调用前保存 input，调用后保存 output 和 token/耗时指标。

In [ ]:
semantic_records: list[dict[str, Any]] = []

class RecordingRunner(SemanticRunner):
    def run(self, contract_name: str, payload: dict[str, Any], **kwargs: Any):  # type: ignore[override]
        index = len(semantic_records) + 1
        input_path = ARTIFACTS_DIR / f'semantic_{index:02d}_{contract_name}_input.json'
        output_path = ARTIFACTS_DIR / f'semantic_{index:02d}_{contract_name}_output.json'
        record_path = ARTIFACTS_DIR / f'semantic_{index:02d}_{contract_name}_record.json'
        input_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
        record: dict[str, Any] = {'index': index, 'contract_name': contract_name, 'input_path': str(input_path), 'output_path': str(output_path), 'kwargs': kwargs}
        started = time.perf_counter()
        try:
            result = super().run(contract_name, payload, **kwargs)
        except Exception as exc:
            record.update({'status': 'failed', 'elapsed_seconds': time.perf_counter() - started, 'error_type': type(exc).__name__, 'error_message': str(exc)})
            semantic_records.append(record)
            record_path.write_text(json.dumps(record, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
            raise
        output = result.output.model_dump(mode='json')
        output_path.write_text(json.dumps(output, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
        record.update({'status': 'ok', 'provider': result.provider, 'model': result.model, 'schema_version': result.schema_version, 'metrics': result.metrics, 'output': output})
        semantic_records.append(record)
        record_path.write_text(json.dumps(record, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
        return result

base_runner = build_semantic_runner(config, provider_name='deepseek')
recording_runner = RecordingRunner(base_runner.client, base_runner.retry_policy)
workflow = IngestSemanticWorkflow(recording_runner)
display(Markdown('RecordingRunner 已准备好。'))

## 5. 运行完整 Ingest

这一格会真实调用 DeepSeek。固定样本通常会调用 5 次模型。

In [ ]:
pipeline = IngestPipeline(workflow)
pipeline_result = pipeline.run(
    config,
    connector_names=['markdown'],
    write=True,
    max_tokens=config.models.resolve_max_tokens('deepseek'),
    write_report=True,
    append_ledger=True,
)
(ARTIFACTS_DIR / 'ingest_pipeline_result.json').write_text(
    json.dumps(pipeline_result.model_dump(mode='json'), ensure_ascii=False, indent=2, default=str),
    encoding='utf-8',
)
(ARTIFACTS_DIR / 'semantic_records.json').write_text(
    json.dumps(semantic_records, ensure_ascii=False, indent=2, default=str),
    encoding='utf-8',
)

display(Markdown('### Pipeline Stats'))
display(JSON(pipeline_result.stats, expanded=True))
display(Markdown('### Pipeline Metrics'))
display(JSON(pipeline_result.metrics, expanded=False))
display(Markdown(f'报告路径：`{pipeline_result.report_path}`'))

## 6. Source 级结果：Checkpoint / Redaction / Segmentation / Lint

这里展示单个来源经过 ingest 后的确定性状态。

In [ ]:
ingest_source = pipeline_result.results[0]
source_summary = {
    'connector': ingest_source.connector,
    'source_id': ingest_source.source_id,
    'source_file': ingest_source.source_file,
    'should_process': ingest_source.should_process,
    'mode': ingest_source.mode,
    'reason': ingest_source.reason,
    'status': ingest_source.status,
    'wrote': ingest_source.wrote,
    'generated_pages': ingest_source.generated_pages,
    'touched_pages': ingest_source.touched_pages,
    'approved_operation_indexes': ingest_source.approved_operation_indexes,
    'checkpoint': ingest_source.checkpoint,
    'redaction': ingest_source.redaction,
    'segmentation': ingest_source.segmentation,
    'segments': ingest_source.segments,
    'quality_gate': ingest_source.quality_gate,
    'scoped_lint': ingest_source.scoped_lint,
    'scoped_lint_result': ingest_source.scoped_lint_result,
}
display(JSON(source_summary, expanded=False))

## 7. Semantic Agents 输入输出总览

每一行对应一次模型调用。`prompt_tokens / completion_tokens / total_tokens / cached` 可以帮助判断成本和缓存命中。

In [ ]:
rows = []
for record in semantic_records:
    metrics = record.get('metrics', {})
    rows.append({
        '#': record['index'],
        'contract': record['contract_name'],
        'status': record['status'],
        'prompt_tokens': metrics.get('prompt_tokens'),
        'completion_tokens': metrics.get('completion_tokens'),
        'total_tokens': metrics.get('total_tokens'),
        'cached_tokens': metrics.get('prompt_cached_tokens') or metrics.get('prompt_cache_hit_tokens'),
        'elapsed_seconds': metrics.get('elapsed_seconds'),
        'input_path': record['input_path'],
        'output_path': record['output_path'],
    })
display(JSON(rows, expanded=True))

## 8. 逐个查看 Agent 输入输出

修改下面的 `agent_index`，可以查看不同 agent：

1. `source_normalize`
2. `wiki_atom_extract`
3. `wiki_page_plan`
4. `wiki_draft_compile`
5. `ingest_draft_review`

In [ ]:
agent_index = 3  # 改成 1-5 查看不同 agent
record = semantic_records[agent_index - 1]
input_payload = json.loads(Path(record['input_path']).read_text(encoding='utf-8'))
output_payload = json.loads(Path(record['output_path']).read_text(encoding='utf-8'))

display(Markdown(f"## Agent {agent_index}: `{record['contract_name']}`"))
display(Markdown('### Input'))
display(JSON(input_payload, expanded=False))
display(Markdown('### Output'))
display(JSON(output_payload, expanded=False))

## 9. 页面计划与审核决策重点查看

这一步把最关键的两个输出抽出来：页面计划决定写什么，审核决策决定能不能写。

In [ ]:
page_plan = next(record['output'] for record in semantic_records if record['contract_name'] == 'wiki_page_plan')
review = next(record['output'] for record in semantic_records if record['contract_name'] == 'ingest_draft_review')

display(Markdown('### Wiki Page Plan Operations'))
display(JSON(page_plan.get('operations', []), expanded=True))
display(Markdown('### Draft Review Decisions'))
display(JSON(review.get('decisions', []), expanded=True))

## 10. 查看最终写入的 Wiki 页面

这里读取临时 vault 里的最终 Markdown。注意：`generated_pages` 是 content-root 相对路径，真实文件在 `vault/pages/` 下。

In [ ]:
CONTENT_ROOT = content_root(VAULT_DIR)
for relative_page in ingest_source.generated_pages:
    page_path = CONTENT_ROOT / relative_page
    display(Markdown(f'## `{relative_page}`'))
    display(Markdown(f'文件：`{page_path}`'))
    text = page_path.read_text(encoding='utf-8')
    display(Markdown(text))

## 11. 查看报告和索引产物

Ingest 完成后会写运行报告、ledger、token ledger 和机器索引。

In [ ]:
report_path = VAULT_DIR / str(pipeline_result.report_path)
display(Markdown(f'报告文件：`{report_path}`'))
display(Markdown(report_path.read_text(encoding='utf-8')))

produced_files = sorted(path.relative_to(VAULT_DIR).as_posix() for path in VAULT_DIR.rglob('*') if path.is_file())
display(Markdown('### Vault 产物清单'))
display(JSON(produced_files, expanded=True))

## 12. 你应该重点观察什么

1. `source_normalize` 是否忠实保留原文主题，而不是过度发挥。
2. `wiki_atom_extract` 的 facts / claims / relations 是否真的比普通摘要更结构化。
3. `wiki_page_plan` 是否把 source digest 和知识页分开。
4. `wiki_draft_compile` 是否把 claims / relations / synthesis 写成页面主体，而不是只写一篇自由作文。
5. `ingest_draft_review` 是否能拒绝证据不足或边界不清的页面。
6. 最终页面的 `canonical_path` 是否是 content-root 相对路径，例如 `个人-AI-Wiki-的维护原则.md`，source digest 是否是 `sources/...`。

这个 notebook 的全部运行产物都在 `RUN_DIR` 下，可以删除，不会影响真实知识库。